In [1]:
print(123)

123


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from google import genai

# 1. Initialize the client
client = genai.Client()

# 2. Call the models service
# response = client.models.generate_content(
#     model="gemini-2.5-pro",  # Or gemini-2.5-pro depending on your preference
#     contents="Your prompt here"
# )
# print(response.text)

In [1]:
def llm(prompt):
    response = client.models.generate_content(
        model="gemini-2.5-flash",  # "gemini-2.5-flash" is great for speed/cost, or use "gemini-2.5-pro" for complex tasks
        contents=prompt
    )
    return response.text

In [5]:
llm('hey whats up?')

"Hey! Not much, just here and ready to chat. What's up with you?"

In [6]:
question = 'I just discorver the course. Can I join now'
answer = llm(question)
print(answer)

That's great! To help me answer whether you can join now, I need a little more information about **which course you're referring to.**

Please tell me:

*   **The name of the course**
*   **Where you found it** (e.g., Coursera, edX, a specific university website, a private platform, etc.)
*   **Any links you might have**

Once I have those details, I can check if it's an evergreen/self-paced course you can enroll in anytime, or if it's a cohort-based course with specific start dates and enrollment periods (in which case you might need to wait for the next session or check for a late enrollment option).


In [7]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [8]:
prompt = f"""   
Your task is to answer questions from the course participants based on the provided context.    

Use the context to find relevant information and provide accurate answers. If the answer is not found in the context,   
respond with "I dont know." 

Question:   
{question}      

Context:    
{context}
"""

In [1]:
answer = llm(prompt)    
print(answer)

NameError: name 'llm' is not defined

In [ ]:
# RAG code process that we will use throughout the workshop
def rag(question):  
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1349

In [ ]:
documents[1345]

{'id': '28991e4581',
 'course': 'machine-learning-zoomcamp',
 'section': 'Miscellaneous',
 'question': 'GitHub 429: Too Many Requests when downloading a CSV via wget / pd.read_csv(url)',
 'answer': 'GitHub rate-limits unauthenticated requests by IP. Workarounds:\n\n- Wait a few minutes and retry.\n- Click the "Raw" button in the browser, save the file manually, then load it from disk.\n- Add a User-Agent header: `wget --user-agent="Mozilla/5.0" <url>`.\n- Switch network or use a VPN.\n\nFor repeatable use, download once and commit the data file (or read from a cached local copy).'}

In [ ]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just doscovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [ ]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'Can I run the course locally instead of Codespaces?']

In [ ]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)

In [ ]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [ ]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [ ]:
results

[{'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course - Can I still join the course after the start date?',
  'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."},
 {'id': 'c842475338',
  'course': 'mlops-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Homework: Just found this course, can I still submit homeworks?',
  'answer': 'To clarify on **late homework submissions**:\n\n- You cannot submit after the homework is scored, as the form is closed.\n- Once the form is closed (i.e., scored), no further submissions are possible.\n- You can check your code against the solution by reviewing the `homework.md` file.\n\nIf the due date has passed but the form is still "O

In [ ]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
search_results = search(question)

1.6 Building the Prompt

The LLM doesn't see our documents unless we pass them in. So we need to build a prompt that includes the user's question and the search results.

When we build AI systems, we usually split the prompt into two parts:

Instructions (also called the system prompt): this tells the LLM how to behave. It never changes, so it's the same for every request.
User prompt: this changes with every request. It carries the actual question and the retrieved context.
We split them because the instructions are fixed and the user prompt is not. Keeping them apart makes the fixed part easy to reuse and the changing part easy to build fresh each time.

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [ ]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [ ]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just doscovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

1.7 The LLM

The last component of our RAG pipeline is the LLM. It takes the prompt we built and generates an answer.

In [ ]:
response = client.models.generate_content(
        model="gemini-2.5-flash",  # "gemini-2.5-flash" is great for speed/cost, or use "gemini-2.5-pro" for complex tasks
        contents=prompt
    )

In [ ]:
response.text[0]

'Y'

In [ ]:
# Native Gemini syntax
prompt_tokens = response.usage_metadata.prompt_token_count
completion_tokens = response.usage_metadata.candidates_token_count
total_tokens = response.usage_metadata.total_token_count

print(prompt_tokens)
print(completion_tokens)
print(total_tokens)

579
237
1469


In [ ]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

# Updated to use Google GenAI SDK usage attributes
cost = (
    response.usage_metadata.prompt_token_count * input_price +
    response.usage_metadata.candidates_token_count * output_price
)

cost

0.00150075

In [ ]:
config = genai.types.GenerateContentConfig(
    system_instruction=INSTRUCTIONS
)

# Just pass the string directly if there's no back-and-forth history yet!
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt, 
    config=config
)

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 25.967769118s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'model': 'gemini-2.5-flash', 'location': 'global'},

In [ ]:
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    # 1. Handle system instructions via the config object
    config = genai.types.GenerateContentConfig(
        system_instruction=instructions
    )

    # 2. Structure the conversation history using explicit SDK types
    message_history = [
        genai.types.Content(
            role="user",
            parts=[genai.types.Part.from_text(text=user_prompt)]
        )
    ]

    # 3. Call the native Gemini service
    response = client.models.generate_content(
        model=model,
        contents=message_history,
        config=config
    )

    # 4. Return the text directly using the Google native helper
    return response.text

Full RAG

In [ ]:
def rag(query, model="gemini-2.5-flash"): # Updated the default model string
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
from IPython.display import Image, display
import base64

# Your flowchart script
mermaid_code = """
flowchart TD
    U([User])

    APP[Application]

    DB[(DB)]
    DOCS[[D1 ... D5]]

    PROMPT[Build Prompt<br/>Question + Context]

    LLM[LLM]

    ANSWER([Answer])

    U -->|Question| APP

    APP -->|Query| DB
    DB -->|Retrieved Data| DOCS
    DOCS --> APP

    APP --> PROMPT
    PROMPT --> LLM

    LLM --> ANSWER
    ANSWER --> U
"""

# Encode the string to match Mermaid's image server requirements
graphbytes = mermaid_code.encode("ascii")
base64_bytes = base64.b64encode(graphbytes)
base64_string = base64_bytes.decode("ascii")

# Display the rendered RAG architecture diagram directly in your notebook
display(Image(url=f"https://mermaid.ink/img/{base64_string}"))

In [ ]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join the course now.

You don't need a confirmation email; you're accepted. You can start learning and submitting homework immediately (while the submission forms are open). The course videos and GitHub materials are available, and you can start whenever you want.

**Important Note regarding Certificates:**
While you can join and follow the course now, if you want to receive a certificate, you must submit your project while submissions are still being accepted and complete the peer-review process within the timeframe of a "live" cohort. You cannot get a certificate for following the course in a purely self-paced mode outside of these active periods.


In [ ]:
rag("How do I get a certificate?")

'To get a certificate, you must fulfill the following conditions:\n\n1.  **Enroll in a "Live" Cohort:** You can only get a certificate if you finish the course with a "live" cohort. Certificates are not awarded for the self-paced mode.\n2.  **Pass the Capstone Project:** You need to pass the Capstone project. (Missing homework is acceptable, but homework is recommended for reinforcing concepts).\n3.  **Peer-Review Capstone Projects:** After submitting your Capstone project, you need to peer-review 3 other capstone projects. This peer-review process can only be done when the course is actively running (after the form is closed and the peer-review list is compiled).\n4.  **Update Your Official Name:** Ensure your official name, as it appears on your identification documents (passport, national ID card, driver\'s license, etc.), is updated in the "Second field" of your course profile (accessible via the "Edit Course Profile" button). This is the name that will appear on your certificate.'

1.8 RAG Helper

In the previous lessons, we built the RAG flow piece by piece - search, then the prompt, then the LLM call. The pipeline works, but every time we want to use it, we need to repeat the same code.

We'll use this code throughout the course, so let's put it into two reusable files:

ingest.py - loading data and building the search index

rag_helper.py - the RAG logic (search, prompt, LLM)

In [11]:
from dotenv import load_dotenv
load_dotenv()

from ingest import load_faq_data, build_index
from rag_helper import RAGBase
from google import genai  # 1. Swapped OpenAI for Google GenAI

documents = load_faq_data()
index = build_index(documents)

# 2. Initialize the native Google GenAI client
# It will automatically find your GEMINI_API_KEY environment variable
gemini_client = genai.Client()

# 3. Pass the Gemini client to your assistant
assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can still join. However, if you wish to receive a certificate, you must submit your project while submissions are still being accepted.


In [12]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=gemini_client,
    instructions=custom_instructions,
)

In [13]:
assistant.rag("How do I get a certificate?")
assistant.rag("Can I still join the course after it started?")

'Yes, you can still join the course after it has started. You can start whenever you want, as the videos and GitHub materials are available.\n\nIf you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

1.9 Data Ingestion

So far, our RAG pipeline loads data and builds the search index at startup. With minsearch, this is fine - our FAQ dataset is small, so indexing takes less than a second. The entire pipeline runs in one process.

This breaks down as the dataset grows. Fetching data takes time - calling APIs, parsing files, cleaning text. With millions of documents, the startup becomes slow. You don't want to wait minutes every time your service restarts.

Minsearch is in-memory. It's a bunch of Python dictionaries bound to the process where it's running. When you stop the process, the data disappears, so you re-index every time you restart. That's wasteful if indexing is slow or the data takes time to prepare.

So we separate ingestion from querying. One process writes the data to a persistent search index. Another process reads from it. The two run independently and only share the index between them.

The index survives restarts, so we ingest once and query as often as we like. This is the ingestion step from data engineering. We move data from its source into a target system the application can use.

You can use any persistent search backend for this, such as Elasticsearch, OpenSearch, or Qdrant. In this module, we use sqlitesearch, a lightweight search library backed by SQLite FTS5. It has the same API as minsearch, so it's a drop-in replacement that happens to be persistent.

I picked SQLite because it asks nothing of you. It ships with Python, so you don't add any dependency, and it has FTS5 (full text search) built in. If you have Python, you already have a full text search engine. Using FTS5 directly is a bit awkward, so I wrote sqlitesearch as a convenient wrapper around it.